# Teste: o Laya roda no Databricks Free Edition?

Teste do professor, não é material de aluno. Responde a três perguntas antes de levar o Laya para o fim do Lab 1:

1. O compute *Serverless* instala o pacote `laya` (PyPI) e baixa o checkpoint `convaiinnovations/laya-multilingual` (Hugging Face, cerca de 1,3 GB)?
2. Quanto tempo leva para carregar e para responder, só com CPU?
3. As probabilidades separam um critério bem escrito de um mal escrito, em português?

Rode as células na ordem e anote o tempo de cada uma.

In [ ]:
%pip install -q "laya==0.3.20"
dbutils.library.restartPython()

In [ ]:
# Baixa o checkpoint para o volume da Blue Blood (uma vez só; nas próximas sessões ele já está lá).
import os, time
os.environ["USE_TF"] = "0"  # o README do Laya avisa que laya.load() pode travar se houver TensorFlow
from huggingface_hub import snapshot_download

PASTA = "/Volumes/blue-blood/bronze/arquivos/modelos/laya-multilingual"
inicio = time.time()
snapshot_download(repo_id="convaiinnovations/laya-multilingual",
                  revision="1720e3e3357cfe1e281542e223f8273b0890ca34",
                  local_dir=PASTA)
print(f"Download: {time.time() - inicio:.0f} s")

In [ ]:
import os, time, json
os.environ["USE_TF"] = "0"
import laya

PASTA = "/Volumes/blue-blood/bronze/arquivos/modelos/laya-multilingual"
inicio = time.time()
agente = laya.load(PASTA, device="cpu")
print(f"Carga: {time.time() - inicio:.0f} s")

In [ ]:
# Mesma pergunta para um critério bom e um ruim. Esperado: noul alto no bom e baixo no ruim.
# State mínimo de propósito: com o Laya, campos vizinhos no state atrapalhavam o julgamento.
questions = {
    "criterio_mensuravel": {
        "type": "noul",
        "instructions": "O critério tem um indicador que pode ser medido com dados e um limiar numérico a partir do qual o comando age?",
    }
}
casos = {
    "bom": "Critério: Evidência. Indicador: variação do roubo de rua no trimestre contra o mesmo trimestre do ano anterior, na CISP, segundo o ISP. Limiar: reforçar o policiamento se a alta passar de 15% e não for sazonal.",
    "ruim": "Critério: Evidência. Indicador: a percepção da população. Limiar: agir quando a situação ficar ruim.",
}
for nome, texto in casos.items():
    inicio = time.time()
    resultado = agente.predict({"criterio": texto}, questions)
    print(f"--- {nome} ({time.time() - inicio:.1f} s)")
    print(json.dumps(resultado, ensure_ascii=False, indent=2, default=str))